In [2]:
!pip install -r requirements.txt

In [1]:
import mlflow
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error

Parquet files downloaded from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet
!mv green_tripdata_2021-01.parquet data/

--2026-03-29 15:31:19--  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.18, 3.163.140.37, 3.163.140.127, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1333519 (1.3M) [binary/octet-stream]
Saving to: ‘green_tripdata_2021-01.parquet.1’

green_tripdata_2021 100%[===================>]   1.27M   404KB/s    in 3.2s    

2026-03-29 15:31:23 (404 KB/s) - ‘green_tripdata_2021-01.parquet.1’ saved [1333519/1333519]



In [6]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet
!mv green_tripdata_2021-02.parquet data/

--2026-03-29 15:31:45--  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.127, 3.163.140.37, 3.163.140.145, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1145679 (1.1M) [binary/octet-stream]
Saving to: ‘green_tripdata_2021-02.parquet’

green_tripdata_2021 100%[===================>]   1.09M  1.01MB/s    in 1.1s    

2026-03-29 15:31:48 (1.01 MB/s) - ‘green_tripdata_2021-02.parquet’ saved [1145679/1145679]



In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1774809293004, experiment_id='3', last_update_time=1774809293004, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, workspace='default'>

In [3]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    
    df = df[df.trip_type == 2]
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [4]:
df_train = read_dataframe('data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('data/green_tripdata_2021-02.parquet')

In [5]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

target = 'duration'

y_train = df_train[target].values
y_val = df_val[target].values


In [6]:
with mlflow.start_run():

    mlflow.set_tag('developer', 'daniel')
    mlflow.log_param('train-data-path', 'data/green_tripdata_2021-01.parquet')
    mlflow.log_param('val-data-path', 'data/green_tripdata_2021-02.parquet')

    alpha = 0.1
    mlflow.log_param('alpha', alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_val)
    
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric('rmse', rmse)

🏃 View run adorable-donkey-967 at: http://localhost:5000/#/experiments/3/runs/d00486e58e6c4f2ab1275b80ed3e28c8
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [7]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

/Users/danielwohlgemuth/anaconda3/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [8]:
train = xgb.DMatrix(X_train, label=y_train)
val = xgb.DMatrix(X_val, label=y_val)

In [9]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag('model', 'xgboost')
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(val, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric('rmse', rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [ ]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    # 'objective': 'reg:linear', # reg:linear is now deprecated in favor of reg:squarederror
    'objective': 'reg:squarederror',
    'seed': 42,
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials())

[0]	validation-rmse:10.82360                                                                                                                                                
[1]	validation-rmse:10.23362                                                                                                                                                
[2]	validation-rmse:9.75228                                                                                                                                                 
[3]	validation-rmse:9.34205                                                                                                                                                 
[4]	validation-rmse:8.99902                                                                                                                                                 
[5]	validation-rmse:8.74502                                                                                                            

After evaluating the first run, the comparison graph showed a lot of lines passing through the lower regions of the parameters. To see if the results could be improved, another experiment with lower parameter values was executed.
![hyperoptimisation](assets/mlflow-experiments-runs-compare-hyper-parameter-optimisation.png)

In [ ]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -5, -2),
    'reg_alpha': hp.loguniform('reg_alpha', -8, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -9, -4),
    'min_child_weight': hp.loguniform('min_child_weight', -3, 3),
    # 'objective': 'reg:linear', # reg:linear is now deprecated in favor of reg:squarederror
    'objective': 'reg:squarederror',
    'seed': 42,
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials())

[0]	validation-rmse:11.26171                                                                                                                                                
[1]	validation-rmse:11.00644                                                                                                                                                
[2]	validation-rmse:10.77051                                                                                                                                                
[3]	validation-rmse:10.54881                                                                                                                                                
[4]	validation-rmse:10.34547                                                                                                                                                
[5]	validation-rmse:10.15976                                                                                                           

The second run didn't show any meaningful improvement or insights into which tuning parameters affect the training the most.
![hyperoptimisation](assets/mlflow-experiments-runs-compare-hyper-parameter-optimisation-2.png)

Picking the best run and rerunning training to capture more details and store the model using autolog.

In [15]:
params = {
    'learning_rate': 0.03661061223133984,
    'max_depth': 4,
    'min_child_weight': 0.0676130545691077,
    'objective': 'reg:squarederror',
    'reg_alpha': 0.007030448836889214,
    'reg_lambda': 0.01815707402586346,
    'seed': 42
}

# Only certain libraries support autolog. See https://mlflow.org/docs/latest/ml/tracking/autolog/#supported-libraries
mlflow.xgboost.autolog()

booster = xgb.train(
    params=params,
    dtrain=train,
    num_boost_round=1000,
    evals=[(val, 'validation')],
    early_stopping_rounds=50
)

2026/04/02 10:08:29 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'b9677cf23ebd4a5bb3e3491cf76c8d2b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow


[0]	validation-rmse:11.31633
[1]	validation-rmse:11.10826
[2]	validation-rmse:10.91281
[3]	validation-rmse:10.71934
[4]	validation-rmse:10.53752
[5]	validation-rmse:10.36829
[6]	validation-rmse:10.19862
[7]	validation-rmse:10.04215
[8]	validation-rmse:9.88898
[9]	validation-rmse:9.75090
[10]	validation-rmse:9.62012
[11]	validation-rmse:9.49510
[12]	validation-rmse:9.37757
[13]	validation-rmse:9.26300
[14]	validation-rmse:9.16449
[15]	validation-rmse:9.06259
[16]	validation-rmse:8.96029
[17]	validation-rmse:8.86449
[18]	validation-rmse:8.78517
[19]	validation-rmse:8.69888
[20]	validation-rmse:8.62147
[21]	validation-rmse:8.54637
[22]	validation-rmse:8.47669
[23]	validation-rmse:8.41091
[24]	validation-rmse:8.36019
[25]	validation-rmse:8.30180
[26]	validation-rmse:8.24470
[27]	validation-rmse:8.20220
[28]	validation-rmse:8.15276
[29]	validation-rmse:8.10807
[30]	validation-rmse:8.06741
[31]	validation-rmse:8.02784
[32]	validation-rmse:7.99070
[33]	validation-rmse:7.95414
[34]	validation-

2026/04/02 10:08:31 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/04/02 10:08:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run thoughtful-conch-23 at: http://localhost:5000/#/experiments/3/runs/b9677cf23ebd4a5bb3e3491cf76c8d2b
🧪 View experiment at: http://localhost:5000/#/experiments/3
